<a href="https://colab.research.google.com/github/Samarbal/Deep-Learning-/blob/main/Loss_Functions%2C_Optimization%2C_and_Gradient_Descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Loss Functions, Optimization, and Gradient Descent


Samar Mohammed Balousha

2320222601

In [ ]:
# import labraries
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

### **Task 1:Loss Functions (Manual vs. Keras)**

In [ ]:
# mean absoulte error loss function
def manual_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

y_true = np.array([10.0, 15.0])
y_pred = np.array([12.0, 14.0])

print(f"Manual MAE: {manual_mae(y_true, y_pred)}")


mae_loss = tf.keras.losses.MeanAbsoluteError()
print(f"Keras MAE: {mae_loss(y_true, y_pred).numpy()}")


Manual MAE: 1.5
Keras MAE: 1.5


The manual MAE implementation exactly matches the Keras built-in loss function

In [ ]:

# define the binray cross entropy function
bce = tf.keras.losses.BinaryCrossentropy()
y_true = [1, 0]

# Case A: correct and confidence
y_pred_A = [0.95, 0.05]
loss_A = bce(y_true, y_pred_A).numpy()

# Case B: wrong
y_pred_B = [0.20, 0.80]
loss_B = bce(y_true, y_pred_B).numpy()

print(f"Loss A: {loss_A}")
print(f"Loss B: {loss_B}")



Loss A: 0.051293306052684784
Loss B: 1.6094379425048828


analysis ♠

- Case B produces a significantly higher loss (1.609) than Case A (0.0513)
- because Binary Cross-Entropy uses a logarithmic penalty ($-\log(\hat{y})$) that grows exponentially as predictions diverge from the true target.
- Unlike MSE, which caps squared errors between 0 and 1, BCE severely penalizes incorrect predictions to force the model to correct its weights rapidly

### **Task 2:Manual Gradient Descent with tf.GradientTape**

In [ ]:
w = tf.Variable(0.0)  # weight
learning_rate = 0.2

# loop

for step in range(15):
  with tf.GradientTape() as tape:
    # loss function
    loss = w**2 - 6*w + 9

  # calculate the gradiant to the loss
  grad = tape.gradient(loss, w)

  # update the weight w = w - lr * grad
  w.assign_sub(learning_rate * grad)

  print(f"Step: {step}, Loss: {loss.numpy()}, Weight: {w.numpy()}")



Step: 0, Loss: 9.0, Weight: 1.2000000476837158
Step: 1, Loss: 3.239999771118164, Weight: 1.9200000762939453
Step: 2, Loss: 1.1663999557495117, Weight: 2.3519999980926514
Step: 3, Loss: 0.4199047088623047, Weight: 2.6112000942230225
Step: 4, Loss: 0.15116596221923828, Weight: 2.7667200565338135
Step: 5, Loss: 0.05441856384277344, Weight: 2.860032081604004
Step: 6, Loss: 0.019591331481933594, Weight: 2.9160192012786865
Step: 7, Loss: 0.007053375244140625, Weight: 2.9496114253997803
Step: 8, Loss: 0.0025396347045898438, Weight: 2.969766855239868
Step: 9, Loss: 0.0009136199951171875, Weight: 2.9818601608276367
Step: 10, Loss: 0.00032901763916015625, Weight: 2.9891161918640137
Step: 11, Loss: 0.00011920928955078125, Weight: 2.993469715118408
Step: 12, Loss: 4.1961669921875e-05, Weight: 2.996081829071045
Step: 13, Loss: 1.621246337890625e-05, Weight: 2.9976491928100586
Step: 14, Loss: 5.7220458984375e-06, Weight: 2.998589515686035


step 0: weight = 0 ;  loss = 0 - (0*6)+9 = 9 ;  gradiant = ( 2 * 0 ) - 6 = - 6 ; new wieght = 0 - ( 0.2 * -6) = 1.2

step 1 : weight = 1.2 ;  loss = (1.2)**2 -  (1.2 *6)+9 =  3.24 ;  gradiant = ( 2 * 1.2 ) - 6 = -3.6   ; new wieght = 1.2 - ( 0.2 * -3.6)  = 1.92

and so on ...

$$\text{Loss}(w) = w^2 - 6w + 9 = (w - 3)^2$$

$$\frac{d}{dw}\text{Loss}(w) = 2w - 6 = 0$$


Solving for $w$:
$$2w = 6 \implies w = 3.0$$


At $w = 3.0$, the theoretical minimum loss value is:$$\text{Loss}(3.0) = (3.0)^2 - 6(3.0) + 9 = 0.0$$

---
Yes, the implementation successfully converges towards the theoretical minimum.

- Initial State: The optimization starts at $w = 0.0$ with an initial loss of $9.0$.   
- Final State (after 15 steps): The weight reaches $w \approx 2.9986$ and the loss drops to $\approx 5.72 \times 10^{-6}$ (virtually $0.0$).

- Because $2.9986$ is extremely close to $3.0$ and the loss approaches $0$, the manual gradient descent algorithm implemented via tf.GradientTape() effectively converges to the exact theoretical minimum.


### **Task 3:Optimizers and Model Compilation**

In [ ]:
# start with simulating the data
np.random.seed(42)
tf.random.set_seed(42)

x_train = np.random.random((500, 5))  # 5 categories , and 500 sample (500 rows , 5 columns ), random numbers between (0 to 1)
y_train = np.random.random((500, 1))

In [ ]:
#builing model architecture
def build_model():
    model = keras.Sequential([
        # hidden layer with 16 cells and ReLU activation function, and 5 input categories
        keras.layers.Dense(units=16, activation='relu', input_shape=(5,)),

        # output layer : Regression
        keras.layers.Dense(units=1)
    ])
    return model


In [ ]:
# Experiment A: SGD Optimizer

print("Experiment A: SGD Optimizer")
model_a = build_model()

# compile the model with SGD
model_a.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss='mean_squared_error',
    metrics=['mae']
)

#train the model for 10 epochs
history_a = model_a.fit(x_train, y_train, epochs=10, batch_size=32, verbose=1)

Experiment A: SGD Optimizer
Epoch 1/10


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.1135 - mae: 0.2779
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1117 - mae: 0.2762 
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1100 - mae: 0.2747 
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1085 - mae: 0.2733 
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1071 - mae: 0.2720 
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1059 - mae: 0.2708 
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1047 - mae: 0.2697 
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1036 - mae: 0.2688 
Epoch 9/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1026 - mae: 0.2679 
Epoch 10/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1018 - mae: 0.2670 


In [ ]:
# Experiment B: Adam Optimizer

print("Experiment B: Adam Optimizer")
model_b = build_model()

# compile the model with Adam optimozer
model_b.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error',
    metrics=['mae']
)

# train the model for 10 epochs
history_b = model_b.fit(x_train, y_train, epochs=10, batch_size=32, verbose=1)

Experiment B: Adam Optimizer
Epoch 1/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.2788 - mae: 0.4351
Epoch 2/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.1791 - mae: 0.3463 
Epoch 3/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1492 - mae: 0.3158 
Epoch 4/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1377 - mae: 0.3052 
Epoch 5/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1286 - mae: 0.2962 
Epoch 6/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1219 - mae: 0.2896 
Epoch 7/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1168 - mae: 0.2847 
Epoch 8/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1127 - mae: 0.2808 
Epoch 9/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1093 - mae: 0.2773 
Epoch 10/10
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.1065 - mae: 0.2743 


In [ ]:
print("Final Comparison (Epoch 10)")

print(f"SGD  Final Loss: {history_a.history['loss'][-1]:.4f} | Final MAE: {history_a.history['mae'][-1]:.4f}")
print(f"Adam Final Loss: {history_b.history['loss'][-1]:.4f} | Final MAE: {history_b.history['mae'][-1]:.4f}")

Final Comparison (Epoch 10)
SGD  Final Loss: 0.1018 | Final MAE: 0.2670
Adam Final Loss: 0.1065 | Final MAE: 0.2743


- In this run, SGD achieved a lower loss (0.1018) and MAE (0.2670) than Adam (Loss: 0.1065, MAE: 0.2743) after 10 epochs.   

- This occurred because SGD was trained with a higher learning rate ($0.01$) compared to Adam ($0.001$), which allowed SGD to take larger steps toward reducing the loss initially.
- Additionally, 10 epochs is too short for Adam's adaptive learning rate mechanism to fully optimize on simple simulated data.   
